# Empirical expansion review

## tl;dr
The September 2026 snapshot expands lobbying history to 2003-2008 and supplies twelve complete PAC half-years. These are source and design inputs, not identified substitution effects. Comment coding remains issue-level and procurement source/linkage gaps remain unresolved.

## Context & Methods
This offline companion reruns the repository's evidence audit and checks the prepared PAC series against its source report periods. It makes no network requests and requires no credentials. Run from this notebook's directory or the repository root using Python 3.

### Key Assumptions
Exact-name LDA links and historical PAC affiliations remain provisional. Native reporting periods are not independent quarters. Repeated issue rows are not separate monetary observations. A missing amount is not zero. Read `docs/substitution-study-redesign.md`, `docs/comment-uptake-pilot.md`, and `docs/procurement-source-reconciliation.md` for source URLs and remaining requirements.

## Data
Inputs are the public CSV snapshots under `data/calibration/first-wave/`. The audit function profiles LDA filings, FEC reports and historical affiliations, the comment corpus and issue pilot, and the GAO worklist. Input file paths remain repository-relative.

In [1]:
import csv
import importlib.util
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'scripts/audit-empirical-expansion.py').is_file(), 'Run inside this repository'

def load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, root / 'scripts' / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

audit = load_script('expansion_audit', 'audit-empirical-expansion.py')
periods = load_script('fec_periods', 'prepare-substitution-fec-periods.py')
findings = audit.audit()

## Results
### Reproduce the prepared series
Only source-marked latest, non-amended reports enter complete half-years. The preparation rejects missing/nonfinite outcomes, gaps, overlaps, and reports straddling half-years. The reform-straddling half-year is retained but excluded from event contrasts.

In [2]:
raw_reports = audit.read('substitution-fec-report-panel.csv')
prepared, excluded = periods.prepare(raw_reports)
assert prepared == audit.read('substitution-fec-halfyear-panel.csv')
print(f'{len(raw_reports)} raw reports; {len(prepared)} complete half-years; {excluded} excluded versions')
for finding in findings:
    print(f"{finding['item']}: {finding['status']}")
    print(finding['evidence'])

78 raw reports; 12 complete half-years; 6 excluded versions
expanded-lda-history: source_only
issueRows=1060; filingUUIDs=412; actors=6; years=2003,2004,2005,2006,2007,2008; observedPrePeriodsByActor=cand-372cc95f9387:9;cand-5d1da86118e5:9;cand-9948f2974958:6;cand-b905d6833296:9;cand-d5522e62fad7:9;cand-f7178708cc78:9
lda-posting-date-anomaly: review_required
postingBeforeCoveredPeriod=3; filingUUIDs=9bd361f7-98e7-46a9-90cd-267ace5ca84c;a93a23e5-9da2-4c18-8625-b41fc0987d06;bcb55688-3d56-4c98-a546-0730eb923bfa
alternate-channel-reports: bounded_affiliation_candidate
reportRows=78; committees=1; affiliationCycles=2004,2006,2008; overlappingPeriods=6; gapsBetweenReports=0; halfYearStraddlingReports=0; missingOutcomeCells=3
alternate-channel-halfyears: observed_outcome_not_effect
completeHalfYears=12; includedReportVersions=72; eventClasses={'pre': 9, 'straddles_event_excluded': 1, 'post': 2}
comment-corpus: source_refreshed
rows=500; uniqueBodies=145; emptyBodies=0
comment-uptake-pilot: i

## Takeaways
The source snapshot contains 1,060 LDA issue rows across 412 filings and six actors. Five actors have nine observed pre-event reporting periods; the sixth has six. Three filings have posting-date anomalies. The single PAC series contains nine pre-event half-years, one excluded event-straddling half-year and two post-event half-years. One PAC and unassigned reform exposure cannot identify a treatment-control substitution effect. The two comment pilot observations have no adjudicated individual-comment links. The GAO worklist has no verified award links.

This notebook verifies reproducible profiling and transformation, not historical coverage, causal identification, independent manual coding, or representative procurement sampling. It does not alter the Java model or promote calibration claims.